# Lab 7: Clustering Techniques

**Tasks:**
1. **K-Means, Mean Shift, DBSCAN**: Customer Segmentation using K-Means. Compare with DBSCAN on datasets with non-spherical shapes.
2. **Gaussian Mixture Models (GMM)**: Anomaly Detection using GMM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, MeanShift, DBSCAN, estimate_bandwidth
from sklearn.mixture import GaussianMixture

# Set plotting style
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Customer Segmentation using K-Means

We will simulate a customer dataset with features like 'Annual Income' and 'Spending Score'. We will then use K-Means to identify distinct customer segments.

In [ ]:
# Generate synthetic customer data (e.g., 5 segments based on income and spending)
X_customers, y_customers = make_blobs(n_samples=300, centers=5, cluster_std=1.0, random_state=42)

# Apply K-Means
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
customer_segments = kmeans.fit_predict(X_customers)

# Visualize the segments
plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_customers[:, 0], X_customers[:, 1], c=customer_segments, cmap='viridis', s=50, alpha=0.7)
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, alpha=0.9, marker='X', label='Centroids')
plt.title('Customer Segmentation with K-Means')
plt.xlabel('Annual Income (Scaled)')
plt.ylabel('Spending Score (Scaled)')
plt.legend()
plt.show()

## 2. Mean Shift Clustering
Mean Shift is a centroid-based algorithm that discovers clusters without requiring the number of clusters to be specified in advance. It finds dense regions of data points.

In [ ]:
# Estimate bandwidth for Mean Shift
bandwidth = estimate_bandwidth(X_customers, quantile=0.2, n_samples=300)

ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
ms.fit(X_customers)
ms_labels = ms.labels_
ms_centers = ms.cluster_centers_

plt.figure(figsize=(8, 5))
plt.scatter(X_customers[:, 0], X_customers[:, 1], c=ms_labels, cmap='plasma', s=50, alpha=0.7)
plt.scatter(ms_centers[:, 0], ms_centers[:, 1], c='red', s=200, alpha=0.9, marker='X', label='Centroids')
plt.title(f'Mean Shift Clustering (Found {len(ms_centers)} clusters)')
plt.xlabel('Annual Income (Scaled)')
plt.ylabel('Spending Score (Scaled)')
plt.legend()
plt.show()

## 3. Comparison: K-Means vs DBSCAN on Non-Spherical Data
K-Means struggles with clusters that are not spherical. DBSCAN, being a density-based algorithm, can identify clusters of arbitrary shapes and is robust to outliers.

In [ ]:
# Generate moon-shaped dataset
X_moons, y_moons = make_moons(n_samples=500, noise=0.05, random_state=42)

# K-Means on moons
kmeans_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_labels = kmeans_moons.fit_predict(X_moons)

# DBSCAN on moons
dbscan = DBSCAN(eps=0.2, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_moons)

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.scatter(X_moons[:, 0], X_moons[:, 1], c=kmeans_labels, cmap='viridis', s=50)
ax1.set_title('K-Means Clustering on Moons (Fails to separate correctly)')

ax2.scatter(X_moons[:, 0], X_moons[:, 1], c=dbscan_labels, cmap='viridis', s=50)
ax2.set_title('DBSCAN Clustering on Moons (Succeeds)')

plt.show()

## 4. Anomaly Detection using Gaussian Mixture Models (GMM)
GMMs can be used for anomaly detection by estimating the density of the data. Instances located in low-density regions (low probability) are flagged as anomalies.

In [ ]:
# Generate a dataset with some "normal" clusters and add some random uniform "anomalies"
np.random.seed(42)
X_normal, _ = make_blobs(n_samples=500, centers=2, cluster_std=1.5, random_state=42)
X_anomalies = np.random.uniform(low=-10, high=10, size=(20, 2))
X_gmm = np.vstack([X_normal, X_anomalies])

# Fit GMM
gmm = GaussianMixture(n_components=2, n_init=10, random_state=42)
gmm.fit(X_gmm)

# Compute the density (log probability) for each sample
densities = gmm.score_samples(X_gmm)

# Define a threshold to identify anomalies (e.g., the lowest 4% density)
density_threshold = np.percentile(densities, 4)
anomalies = X_gmm[densities < density_threshold]

# Plotting
plt.figure(figsize=(10, 6))
plt.scatter(X_gmm[:, 0], X_gmm[:, 1], c='blue', s=20, alpha=0.5, label='Normal Data')
plt.scatter(anomalies[:, 0], anomalies[:, 1], c='red', s=100, marker='o', edgecolors='black', label='Anomalies')

plt.title('Anomaly Detection using GMM (Identified Low Density Points)')
plt.legend()
plt.show()